In [ ]:
# === CORE DATA + NLP DEPENDENCIES ===
!pip install -q pandas numpy pyarrow fastparquet tqdm

# For text cleaning, regex, etc. (standard in Python) – no extra install needed

# === NLP PIPELINE ===
!pip install -q spacy

# Download the small English model for sentence segmentation + NER
#!python -m spacy download en_core_web_sm -q


In [ ]:
# ============================================
# [A] Mount Drive and define paths
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# ---- BASE PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"   # your project base folder

# ---- INPUTS ----
FORUM_CSV_DRIVE = f"{BASE}/raw/forums/multi_forum_property_posts_hwz_ts.csv"
FORUM_CSV_UPLOADED = "/mnt/data/multi_forum_property_posts_hwz_ts.csv"  # <- prefer if present

# Singlish lexicon (rich version)
SINGLEX_CSV = f"{BASE}/corpus/Singlish/lexicon.csv"

# Property domain resources (SGPropertyDomain)
ENTITYRULER = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
REGEX_JSONL = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"

# ---- OUTPUTS ----
OUTDIR = f"{BASE}/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025"


from pathlib import Path
import pandas as pd
import os, re, json, html, unicodedata

# ============================================
# Auto-merge SGPropertyDomain vocab/*.txt → EntityRuler patterns
# (renamed variables to avoid clobbering BASE)
# ============================================
DOMAIN_BASE = f"{BASE}/corpus/SGPropertyDomain"
VOC_DIR  = f"{DOMAIN_BASE}/vocab"                                  # contains *.txt like HDB.txt, Finance&Rates.txt, ...
EXISTING = f"{DOMAIN_BASE}/spacy_entityruler_patterns.jsonl"       # your current file (ok if missing)
MERGED   = f"{DOMAIN_BASE}/spacy_entityruler_patterns.merged.jsonl" # output we will generate

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


def iter_existing(path):
    out = []
    p = Path(path)
    if not p.exists():
        return out
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except:
                pass
    return out

def phrase_to_token_pattern(phrase: str):
    phrase = re.sub(r"\s+", " ", phrase).strip()
    if not phrase:
        return None
    tokens = phrase.split(" ")
    return [{"LOWER": t.lower()} for t in tokens if t]

def label_from_filename(fname: str):
    stem  = Path(fname).stem
    label = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_").upper()
    return label or "DOMAIN"

# 1) Existing patterns
existing = iter_existing(EXISTING)
print(f"[INFO] Loaded existing patterns: {len(existing)}")

# 2) Build from vocab/*.txt
voc_path = Path(VOC_DIR)
assert voc_path.exists(), f"Vocab directory not found: {VOC_DIR}"

vocab_patterns = []
for txt in sorted(voc_path.glob("*.txt")):
    label = label_from_filename(txt.name)
    for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
        term = raw.strip()
        if not term:
            continue
        pat = phrase_to_token_pattern(term)
        if not pat:
            continue
        vocab_patterns.append({"label": label, "pattern": pat, "id": term})

print(f"[INFO] Built vocab patterns: {len(vocab_patterns)}")

def lowers_from_pattern(pat):
    if isinstance(pat, str):
        return tuple(p.strip().lower() for p in re.sub(r"\s+", " ", pat).split(" ") if p.strip())
    if isinstance(pat, dict):
        return (str(pat.get("LOWER", pat.get("TEXT", ""))).lower(),)
    if isinstance(pat, list):
        outs = []
        for tok in pat:
            if isinstance(tok, dict):
                outs.append(str(tok.get("LOWER", tok.get("TEXT", ""))).lower())
            else:
                outs.append(str(tok).lower())
        return tuple(outs)
    return (str(pat).lower(),)

def pat_key(rec):
    pat = rec.get("pattern", "")
    label = rec.get("label", "")
    return (label, lowers_from_pattern(pat))

seen, merged = set(), []
str_count = list_count = dict_count = 0

for rec in existing:
    if isinstance(rec.get("pattern", ""), str): str_count += 1
    elif isinstance(rec.get("pattern", ""), list): list_count += 1
    elif isinstance(rec.get("pattern", ""), dict): dict_count += 1
    k = pat_key(rec)
    if k in seen:
        continue
    seen.add(k)
    merged.append(rec)

skipped = 0
for rec in vocab_patterns:
    k = pat_key(rec)
    if k in seen:
        skipped += 1
        continue
    seen.add(k)
    merged.append(rec)

print(f"[INFO] Existing pattern types — str:{str_count}, list:{list_count}, dict:{dict_count}")
print(f"[INFO] Merged total patterns: {len(merged)} (skipped dupes: {skipped})")

with open(MERGED, "w", encoding="utf-8") as f:
    for rec in merged:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"[DONE] Wrote merged patterns → {MERGED}")

# point pipeline to merged file
ENTITYRULER = MERGED
print("ENTITYRULER now set to:", ENTITYRULER)

# ============================================
# Helpers
# ============================================
def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def basic_clean(s: str) -> str:
    if not isinstance(s, str): return ""
    s = html.unescape(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"(https?://\S+|www\.\S+)", " ", s)
    s = re.sub(r"[\[\]{}<>]", " ", s)
    return normalize_ws(s)

def load_jsonl(path: Path):
    items=[]
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if ln:
                try: items.append(json.loads(ln))
                except: pass
    return items

def compile_regexes_from_jsonl(path: Path):
    patt=[]
    for it in load_jsonl(path):
        pat = it.get("pattern")
        if not pat: continue
        try: patt.append({"name": it.get("name","pattern"), "re": re.compile(pat)})
        except re.error: pass
    return patt

def detect_regex_hits(text: str, compiled):
    hits={}
    for p in compiled:
        try:
            if p["re"].search(text): hits[p["name"]] = True
        except: pass
    return hits

# ---- Singlish loaders ----
def build_singdict_from_lexicon(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "word" not in df.columns:
        raise ValueError(f"'word' column not found in {csv_path}. Columns: {df.columns.tolist()}")
    df = df.dropna(subset=["word"])
    words_set = set()
    meta_map  = {}
    for _, r in df.iterrows():
        w = str(r["word"]).strip().lower()
        if not w: continue
        words_set.add(w)
        desc = str(r.get("description", "")).strip()
        meta_map[w] = {"description": desc} if desc else {}
    return words_set, meta_map

def find_singlish_terms(text: str, words_set):
    if not words_set: return [], text
    toks = re.findall(r"[A-Za-z][A-Za-z\-']+|\d+|[^\w\s]", text)
    found, out = [], []
    for tok in toks:
        low = tok.lower()
        if low in words_set:
            found.append(low); out.append(low)
        else:
            out.append(tok)
    return sorted(list(set(found))), normalize_ws(" ".join(out))

def add_entity_ruler_spacy(df: pd.DataFrame, text_col: str, patterns_path: str):
    try:
        import spacy
        nlp = spacy.blank("en")
        ruler = nlp.add_pipe("entity_ruler")
        ruler.from_disk(str(patterns_path))
        ents=[]
        for doc in nlp.pipe(df[text_col].astype(str).tolist(), batch_size=64):
            ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
        df["entities"] = ents
    except Exception as e:
        print(f"[WARN] spaCy entity_ruler skipped: {e}")
        df["entities"] = [[] for _ in range(len(df))]
    return df

# ============================================
# Load forum data (prefer uploaded file if present)
# ============================================
forum_path = FORUM_CSV_UPLOADED if Path(FORUM_CSV_UPLOADED).exists() else FORUM_CSV_DRIVE
assert Path(forum_path).exists(), f"Forum file missing: {forum_path}"
assert Path(SINGLEX_CSV).exists(), f"Lexicon file missing: {SINGLEX_CSV}"

df = pd.read_csv(forum_path)

print(df.columns.tolist())
print(df.head(3).to_dict(orient="records"))
print(df["thread_url"].head(10).tolist() if "thread_url" in df.columns else "No thread_url col")

# ============================================
# 1) LOAD + NORMALIZE COLUMNS
# ============================================
TEXT_COL_CANDS = ["post_text", "text", "content", "body"]
TEXT_COL = next((c for c in TEXT_COL_CANDS if c in df.columns), None)
assert TEXT_COL is not None, f"No text column found. Columns: {list(df.columns)}"
if TEXT_COL != "post_text":
    df = df.rename(columns={TEXT_COL: "post_text"})

if "forum_name" not in df.columns and "forum" in df.columns:
    df = df.rename(columns={"forum": "forum_name"})

# ---- Build a 'date' column (we WILL enforce year filter) ----
DATE_COL_CANDS = ["date", "posted_at", "post_date", "created_at", "timestamp", "time"]
DATE_COL = next((c for c in DATE_COL_CANDS if c in df.columns), None)

def try_parse_dt(x):
    return pd.to_datetime(x, errors="coerce")

def extract_date_from_url_like(s: str):
    if not isinstance(s, str):
        return pd.NaT
    m = re.search(r"(20\d{2})-(0?[1-9]|1[0-2])-(0?[1-9]|[12]\d|3[01])", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
    m = re.search(r"(20\d{2})/(0?[1-9]|1[0-2])/(0?[1-9]|[12]\d|3[01])", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
    m = re.search(r"(20\d{2})_(0?[1-9]|1[0-2])_(0?[1-9]|[12]\d|3[01])", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
    m = re.search(r"\b(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])\b", s)
    if m:
        return try_parse_dt(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")
    # UNIX epoch seconds (10) or ms (13)
    m = re.search(r"(?<!\d)(1[5-9]\d{8}|2\d{9})(?!\d)", s)
    if m:
        return pd.to_datetime(int(m.group(1)), unit="s", errors="coerce", utc=True).tz_localize(None)
    m = re.search(r"(?<!\d)(1[5-9]\d{11}|2\d{12})(?!\d)", s)
    if m:
        return pd.to_datetime(int(m.group(1)), unit="ms", errors="coerce", utc=True).tz_localize(None)
    return pd.NaT

if DATE_COL is not None:
    df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
else:
    url_cols = [c for c in ["thread_url", "post_url", "url", "source_url"] if c in df.columns]
    if url_cols:
        dates = pd.Series(pd.NaT, index=df.index)
        for c in url_cols:
            cand = df[c].astype(str).apply(extract_date_from_url_like)
            dates = dates.fillna(cand)
        df["date"] = dates
    else:
        df["date"] = pd.NaT

# ============================================
# >>> STRICT YEAR FILTER: keep ONLY 2023–2025 <<<
# ============================================
YEAR_MIN, YEAR_MAX = 2023, 2025
before_year_filter = len(df)

# require non-null date, then filter inclusive range
df = df.dropna(subset=["date"]).copy()
df = df[(df["date"].dt.year >= YEAR_MIN) & (df["date"].dt.year <= YEAR_MAX)].copy()

after_year_filter = len(df)
print(f"[INFO] Year filter {YEAR_MIN}-{YEAR_MAX}: kept {after_year_filter} / {before_year_filter} rows (dropped {before_year_filter - after_year_filter})")

HAS_DATE = df["date"].notna().any()
print(f"[INFO] HAS_DATE (post-filter) = {HAS_DATE}  (non-null dates: {df['date'].notna().sum()})")

# ============================================
# 2) CLEAN + DEDUP (after year filter)
# ============================================
df["raw_text"]   = df["post_text"].astype(str)
df["clean_text"] = df["raw_text"].apply(basic_clean)
df = df[df["clean_text"].str.len() > 20].copy()

key_parts = [df["clean_text"].astype(str)]
if "author" in df.columns:
    key_parts.insert(0, df["author"].astype(str))
if "title" in df.columns:
    key_parts.insert(0, df["title"].astype(str))
key_parts.append(df["date"].dt.date.astype(str))  # date exists & is within range

df["_k"] = key_parts[0]
for col in key_parts[1:]:
    df["_k"] = df["_k"] + "||" + col

before = len(df)
df = df.drop_duplicates(subset=["_k"]).drop(columns=["_k"])
after = len(df)
print(f"[INFO] Dedupe removed {before - after} rows  →  kept {after}")

# ============================================
# 3) SINGLISH + PROPERTY ENRICHMENT
# ============================================
sing_words, sing_meta = build_singdict_from_lexicon(SINGLEX_CSV)
found_terms, norm_texts, meanings = [], [], []
for t in df["raw_text"].astype(str):
    f, n = find_singlish_terms(t, sing_words)
    found_terms.append(f)
    norm_texts.append(n)
    m = [sing_meta[w]["description"] for w in f if w in sing_meta and sing_meta[w].get("description")]
    meanings.append(list(dict.fromkeys(m)))
df["singlish_terms"]    = found_terms
df["has_singlish"]      = df["singlish_terms"].apply(bool)
df["text_sing_norm"]    = norm_texts
df["singlish_meanings"] = meanings

if REGEX_JSONL and Path(REGEX_JSONL).exists():
    comp = compile_regexes_from_jsonl(Path(REGEX_JSONL))
    hits = [detect_regex_hits(t, comp) for t in df["clean_text"].astype(str)]
    hdf  = pd.json_normalize(hits)
    hdf.columns = [f"rx_{c}" for c in hdf.columns]
    df = pd.concat([df.reset_index(drop=True), hdf.reset_index(drop=True)], axis=1)

if ENTITYRULER and Path(ENTITYRULER).exists():
    df = add_entity_ruler_spacy(df, "clean_text", ENTITYRULER)
else:
    df["entities"] = [[] for _ in range(len(df))]

df["num_chars"] = df["clean_text"].str.len()
df["num_words"] = df["clean_text"].str.split().apply(len)
print(f"[STATS] total rows: {len(df)} | posts with Singlish: {df['has_singlish'].sum()}")

# ============================================
# 4) SAVE CORE
# ============================================
Path(OUTDIR).mkdir(parents=True, exist_ok=True)
out_csv  = f"{OUTDIR}/forum_enriched.csv"
out_parq = f"{OUTDIR}/forum_enriched.parquet"
df.to_csv(out_csv, index=False)
try:
    df.to_parquet(out_parq, index=False)
except Exception as e:
    print(f"[WARN] Parquet write failed: {e}")

stats = {
    "rows_after_year_filter": int(after_year_filter),
    "rows_after_clean_dedup": int(len(df)),
    "dropped_before_year_filter": int(before_year_filter - after_year_filter),
    "has_singlish_true": int(df["has_singlish"].sum()),
    "cols": list(df.columns),
    "has_date": bool(HAS_DATE),
    "non_null_dates": int(df["date"].notna().sum()),
    "year_min": YEAR_MIN,
    "year_max": YEAR_MAX,
    "source_file": forum_path,
}
with open(f"{OUTDIR}/preprocess_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print("Saved:", out_csv)
print("Saved:", out_parq)
print("Saved:", f"{OUTDIR}/preprocess_stats.json")




[INFO] Loaded existing patterns: 2256
[INFO] Built vocab patterns: 1025
[INFO] Existing pattern types — str:2256, list:0, dict:0
[INFO] Merged total patterns: 2158 (skipped dupes: 795)
[DONE] Wrote merged patterns → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl
ENTITYRULER now set to: /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl
['forum', 'thread_url', 'post_text']
[{'forum': 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'thread_url': 'Not sure if this been posted or not ? hdb resales transactions private ppty transactions', 'post_text': '2003-07-11T12:42:09+0800'}, {'forum': 'https://forums.hardwarezone.com.sg/threads/useful-websites.511156/', 'thread_url': "i have posted few times on the hdb resale transaction's link bt not the pte ppty one. will put it as sticky. =p", 'post_text': '2003-07-11T15:37:04+0800'}, {'forum': 'https://forums.hardwarezone.com.sg

In [ ]:
# ============================================
# 5) NLP ENRICHMENT (tokens/lemmas/NER/ABSA scaffolding + temporal trends)
# ============================================


import spacy
nlp = spacy.load("en_core_web_sm", exclude=[])  # tagger, parser, NER
try:
    if ENTITYRULER and Path(ENTITYRULER).exists():
        ruler = nlp.add_pipe("entity_ruler", before="ner")
        ruler.from_disk(ENTITYRULER)
except Exception as e:
    print("[WARN] EntityRuler reattach skipped:", e)

docs = list(nlp.pipe(df["clean_text"].astype(str).tolist(), batch_size=64, n_process=2))

df["tokens"] = [[t.text  for t in d] for d in docs]
df["lemmas"] = [[t.lemma_ for t in d] for d in docs]
df["pos"]    = [[t.pos_   for t in d] for d in docs]
df["deps"]   = [[t.dep_   for t in d] for d in docs]

# sentence table (dates guaranteed present & in-range)
sent_rows = []
for i, d in enumerate(docs):
    for j, s in enumerate(d.sents):
        sent_rows.append({
            "post_id": i,
            "sent_id": j,
            "text": s.text,
            "tokens": [t.text for t in s],
            "lemmas": [t.lemma_ for t in s],
            "pos":    [t.pos_   for t in s],
            "deps":   [t.dep_   for t in s],
            "date":   df.iloc[i]["date"]
        })

sent_df = pd.DataFrame(sent_rows)
sent_path = f"{OUTDIR}/forum_sentences.parquet"
sent_df.to_parquet(sent_path, index=False)
print("Saved sentence table →", sent_path)

df["entities_ner"] = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in docs]
df["aspect_candidates"] = [[nc.text for nc in d.noun_chunks] for d in docs]
df.to_parquet(f"{OUTDIR}/forum_enriched+nlp.parquet", index=False)
print("Saved:", f"{OUTDIR}/forum_enriched+nlp.parquet")

# monthly trend (always true now since we filtered by year + dropped NaT)
monthly = (
    df.assign(month=df["date"].dt.to_period("M").astype(str))
      .groupby("month", dropna=True)
      .agg(posts=("clean_text", "size"),
           with_singlish=("has_singlish", "sum"))
      .reset_index()
)
monthly.to_csv(f"{OUTDIR}/monthly_counts.csv", index=False)
print("Saved:", f"{OUTDIR}/monthly_counts.csv")

# quick checks
print("OUTDIR:", OUTDIR)
print("Files:", sorted(os.listdir(OUTDIR))[:12])

enriched_csv = f"{OUTDIR}/forum_enriched.csv"
if os.path.exists(enriched_csv):
    df_check = pd.read_csv(enriched_csv, nrows=3)
    print("\n✅ Loaded:", enriched_csv)
    print("Columns:", df_check.columns.tolist())
else:
    print("\n❌ forum_enriched.csv not found!")

print("Has monthly_counts.csv?", os.path.exists(f"{OUTDIR}/monthly_counts.csv"))
print("Has sentence parquet?", os.path.exists(sent_path))
print("Has enriched+nlp parquet?", os.path.exists(f"{OUTDIR}/forum_enriched+nlp.parquet"))
stats_path = f"{OUTDIR}/preprocess_stats.json"
print("Has stats json?", os.path.exists(stats_path))
if os.path.exists(stats_path):
    print("\nStats JSON contents:")
    print(json.load(open(stats_path)))

Saved sentence table → /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_sentences.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched+nlp.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/monthly_counts.csv
OUTDIR: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025
Files: ['forum_enriched+nlp.parquet', 'forum_enriched.csv', 'forum_enriched.parquet', 'forum_sentences.parquet', 'monthly_counts.csv', 'preprocess_stats.json']

✅ Loaded: /content/drive/MyDrive/PropInsight/preprocess/forum/multi_forum_property_posts_hwz_processed_2023_2025/forum_enriched.csv
Columns: ['forum_name', 'thread_url', 'post_text', 'date', 'raw_text', 'clean_text', 'singlish_terms', 'has_singlish', 'text_sing_norm', 'singlish_meanings', 'entities', 'num_chars